# 04 - Models: Deep Neural Network (DNN) con TF-IDF

Este notebook entrena un modelo **Deep Neural Network (DNN)** para la familia `04_models_<miembro>` usando la codificacion **TF-IDF** generada en `02_encoding.ipynb`.

El entrenamiento usa **todo el dataset** y trabaja con mini-batches sobre la matriz sparse TF-IDF para no densificar todo el corpus en memoria. El flujo mantiene el mismo patron de evaluacion y registro en **MLflow** del notebook `03_baseline.ipynb`.

## 1. Instalacion e imports

In [ ]:
!pip install -q torch scikit-learn scipy joblib matplotlib seaborn mlflow

In [ ]:
import copy
import json
import logging
import warnings
from pathlib import Path

import joblib
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from scipy.sparse import load_npz
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset

SEED = 42
MEMBER = "asantiagodiaz"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
print('Device:', DEVICE)

## 2. Carga de artefactos

In [ ]:
X_tr = load_npz('X_tr.npz')
X_val = load_npz('X_val.npz')
X_train_tfidf = load_npz('X_train_tfidf.npz')
X_test_tfidf = load_npz('X_test_tfidf.npz')

y_tr = joblib.load('y_tr.pkl')
y_val = joblib.load('y_val.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

print('Shapes cargados:')
print('  X_tr          :', X_tr.shape)
print('  X_val         :', X_val.shape)
print('  X_train_tfidf :', X_train_tfidf.shape)
print('  X_test_tfidf  :', X_test_tfidf.shape)
print('  y_tr          :', len(y_tr))
print('  y_val         :', len(y_val))
print('  y_train       :', len(y_train))
print('  y_test        :', len(y_test))

## 3. Entrenamiento con todo el dataset

Este notebook usa **todo el dataset disponible en los artefactos de `02_encoding.ipynb`**. `X_tr` se usa completo para seleccionar hiperparametros, `X_val` completo para validacion, y el modelo final se reentrena sobre `X_train_tfidf` completo.

In [ ]:
print('Configuracion de entrenamiento:')
print('  Tuning train :', X_tr.shape)
print('  Validacion   :', X_val.shape)
print('  Train final  :', X_train_tfidf.shape)
print('  Test final   :', X_test_tfidf.shape)
print('  balance y_tr   :', np.bincount(y_tr))
print('  balance y_val  :', np.bincount(y_val))
print('  balance y_train:', np.bincount(y_train))
print('  balance y_test :', np.bincount(y_test))

## 4. Utilidades del DNN

In [ ]:
BATCH_SIZE = 128

class SparseTextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.tocsr()
        self.y = np.asarray(y, dtype=np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def sparse_collate_fn(batch):
    rows, labels = zip(*batch)
    X_batch = np.vstack([row.toarray() for row in rows]).astype(np.float32)
    y_batch = np.asarray(labels, dtype=np.float32)
    return torch.from_numpy(X_batch), torch.from_numpy(y_batch).unsqueeze(1)

def make_loader(X, y, batch_size=BATCH_SIZE, shuffle=False):
    dataset = SparseTextDataset(X, y)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=sparse_collate_fn,
    )

class SentimentDNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)
    return running_loss / len(loader.dataset)

def predict_proba(model, loader, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            batch_probs = torch.sigmoid(logits).cpu().numpy().ravel()
            probs.append(batch_probs)
    return np.concatenate(probs)

def compute_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'f1_macro': round(f1_score(y_true, y_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_true, y_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_true, y_pred, average='macro'), 4),
        'roc_auc': round(roc_auc_score(y_true, y_proba), 4),
    }

def fit_dnn(config, X_fit, y_fit, X_eval, y_eval, device):
    train_loader = make_loader(X_fit, y_fit, batch_size=config['batch_size'], shuffle=True)
    eval_loader = make_loader(X_eval, y_eval, batch_size=config['batch_size'], shuffle=False)

    model = SentimentDNN(
        input_dim=X_fit.shape[1],
        hidden_dims=config['hidden_dims'],
        dropout=config['dropout'],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.BCEWithLogitsLoss()

    history = []
    best_state = None
    best_val_f1 = -1
    best_epoch = 0
    patience_left = config['patience']

    for epoch in range(1, config['epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_proba = predict_proba(model, eval_loader, device)
        val_metrics = compute_metrics(y_eval, val_proba)

        history.append({
            'epoch': epoch,
            'train_loss': round(train_loss, 4),
            **val_metrics,
        })

        if val_metrics['f1_macro'] > best_val_f1:
            best_val_f1 = val_metrics['f1_macro']
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = config['patience']
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


## 5. Busqueda de hiperparametros

In [ ]:
param_grid = [
    {
        'hidden_dims': [128, 64],
        'dropout': 0.30,
        'lr': 1e-3,
        'batch_size': 128,
        'epochs': 5,
        'patience': 2,
    },
    {
        'hidden_dims': [256, 64],
        'dropout': 0.35,
        'lr': 8e-4,
        'batch_size': 128,
        'epochs': 5,
        'patience': 2,
    },
]

results_dnn = []
histories = {}
best_model = None
best_params = None
best_history = None
best_epoch = None
best_f1 = -1

for idx, params in enumerate(param_grid, start=1):
    print(f'Entrenando configuracion {idx}/{len(param_grid)}: {params}')
    model, history_df, selected_epoch = fit_dnn(params, X_tr, y_tr, X_val, y_val, DEVICE)
    val_proba = predict_proba(model, make_loader(X_val, y_val, batch_size=params['batch_size']), DEVICE)
    metrics_row = {
        **params,
        'best_epoch': selected_epoch,
        **compute_metrics(y_val, val_proba),
    }
    results_dnn.append(metrics_row)
    histories[f'cfg_{idx}'] = history_df

    if metrics_row['f1_macro'] > best_f1:
        best_f1 = metrics_row['f1_macro']
        best_params = params
        best_model = model
        best_history = history_df
        best_epoch = selected_epoch

df_dnn = pd.DataFrame(results_dnn).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df_dnn)
print('Mejores hiperparametros:', best_params)
print('Best epoch:', best_epoch)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df_dnn.copy()
plot_df['config'] = [f"{r.hidden_dims} | drop={r.dropout}" for r in plot_df.itertuples()]
ax.bar(plot_df['config'], plot_df['f1_macro'], color='steelblue')
ax.set_title('DNN - F1 macro en validacion')
ax.set_xlabel('Configuracion')
ax.set_ylabel('F1 macro')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(best_history['epoch'], best_history['train_loss'], marker='o', label='Train loss')
ax.plot(best_history['epoch'], best_history['f1_macro'], marker='s', label='Val F1 macro')
ax.set_title('Mejor configuracion DNN')
ax.set_xlabel('Epoch')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Entrenamiento final

In [ ]:
final_params = {**best_params, 'epochs': best_epoch, 'patience': best_epoch}
model_final, train_history_final, _ = fit_dnn(final_params, X_train_tfidf, y_train, X_val, y_val, DEVICE)

print('Modelo final entrenado con todo el train.')
print('  Train final shape:', X_train_tfidf.shape)
print('  hidden_dims      :', best_params['hidden_dims'])
print('  dropout          :', best_params['dropout'])
print('  lr               :', best_params['lr'])
print('  best_epoch       :', best_epoch)

## 7. Evaluacion final en test

In [ ]:
test_loader = make_loader(X_test_tfidf, y_test, batch_size=best_params['batch_size'], shuffle=False)
y_proba = predict_proba(model_final, test_loader, DEVICE)
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')

print('Metricas en test:')
print(f'  Accuracy   : {acc:.4f}')
print(f'  F1 macro   : {f1:.4f}')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  ROC AUC    : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo (0)', 'Positivo (1)'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('DNN + TF-IDF')
plt.tight_layout()
plt.show()

## 8. Guardar modelo y resultados

In [ ]:
model_path = 'dnn_tfidf_model.pt'
config_path = 'dnn_tfidf_config.json'

torch.save(model_final.state_dict(), model_path)

config_to_save = {
    'input_dim': int(X_train_tfidf.shape[1]),
    'hidden_dims': best_params['hidden_dims'],
    'dropout': best_params['dropout'],
    'lr': best_params['lr'],
    'batch_size': best_params['batch_size'],
    'best_epoch': int(best_epoch),
}

Path(config_path).write_text(json.dumps(config_to_save, indent=2))

results_final = pd.DataFrame([{
    'modelo': 'Deep Neural Network',
    'encoding': 'TF-IDF',
    'member': MEMBER,
    'train_size_tuning': X_tr.shape[0],
    'val_size': X_val.shape[0],
    'final_train_size': len(y_train),
    'hidden_dims': str(best_params['hidden_dims']),
    'dropout': best_params['dropout'],
    'lr': best_params['lr'],
    'batch_size': best_params['batch_size'],
    'best_epoch': best_epoch,
    'accuracy': round(acc, 4),
    'f1_macro': round(f1, 4),
    'roc_auc': round(auc, 4)
}])

results_final.to_csv('results_dnn_tfidf.csv', index=False)

print('Archivos guardados:')
print('  dnn_tfidf_model.pt    -> pesos del modelo DNN')
print('  dnn_tfidf_config.json -> configuracion del modelo')
print('  results_dnn_tfidf.csv -> tabla de metricas')
print()
print(results_final.to_string(index=False))

## 9. Registro en MLflow

In [ ]:
import logging
import warnings

logging.getLogger('mlflow.tracking.request_header.registry').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*pickle or cloudpickle format.*', category=FutureWarning)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'DNN_TFIDF_{MEMBER}') as run:
    mlflow.set_tags({
        'user': 'Andres Santiago Diaz',
        'member': MEMBER,
        'model_type': 'DeepNeuralNetwork',
        'encoding': 'TF-IDF',
        'dataset': 'Sentiment140Twitter',
    })

    mlflow.log_params({
        'prep_remove_urls': True,
        'prep_remove_mentions': True,
        'prep_remove_hashtags': True,
        'prep_remove_emojis': True,
        'prep_remove_stopwords': False,
        'prep_lemmatization': False,
    })

    mlflow.log_params({
        'vec_type': 'TfidfVectorizer',
        'vec_max_features': 100000,
        'vec_min_df': 5,
        'vec_max_df': 0.95,
        'vec_ngram_range': '(1,2)',
        'vec_sublinear_tf': True,
    })

    mlflow.log_params({
        'model': 'DNN',
        'hidden_dims': str(best_params['hidden_dims']),
        'dropout': best_params['dropout'],
        'lr': best_params['lr'],
        'batch_size': best_params['batch_size'],
        'best_epoch': best_epoch,
        'train_size_tuning': X_tr.shape[0],
        'val_size': X_val.shape[0],
        'final_train_size': X_train_tfidf.shape[0],
        'test_size': X_test_tfidf.shape[0],
        'vocab_size': X_train_tfidf.shape[1],
        'device': str(DEVICE),
    })

    train_loader_eval = make_loader(X_train_tfidf, y_train, batch_size=best_params['batch_size'], shuffle=False)
    y_proba_train = predict_proba(model_final, train_loader_eval, DEVICE)
    y_pred_train = (y_proba_train >= 0.5).astype(int)
    mlflow.log_metrics({
        'train_accuracy': round(accuracy_score(y_train, y_pred_train), 4),
        'train_f1_macro': round(f1_score(y_train, y_pred_train, average='macro'), 4),
        'train_precision': round(precision_score(y_train, y_pred_train, average='macro'), 4),
        'train_recall': round(recall_score(y_train, y_pred_train, average='macro'), 4),
        'train_roc_auc': round(roc_auc_score(y_train, y_proba_train), 4),
    })

    val_loader_eval = make_loader(X_val, y_val, batch_size=best_params['batch_size'], shuffle=False)
    y_proba_val = predict_proba(best_model, val_loader_eval, DEVICE)
    y_pred_val = (y_proba_val >= 0.5).astype(int)
    mlflow.log_metrics({
        'val_accuracy': round(accuracy_score(y_val, y_pred_val), 4),
        'val_f1_macro': round(f1_score(y_val, y_pred_val, average='macro'), 4),
        'val_precision': round(precision_score(y_val, y_pred_val, average='macro'), 4),
        'val_recall': round(recall_score(y_val, y_pred_val, average='macro'), 4),
        'val_roc_auc': round(roc_auc_score(y_val, y_proba_val), 4),
    })

    mlflow.log_metrics({
        'test_accuracy': round(acc, 4),
        'test_f1_macro': round(f1, 4),
        'test_precision': round(precision, 4),
        'test_recall': round(recall, 4),
        'test_roc_auc': round(auc, 4),
    })

    mlflow.log_artifact('results_dnn_tfidf.csv')
    mlflow.log_artifact(config_path)
    mlflow.log_artifact(model_path)
    mlflow.pytorch.log_model(
        model_final.cpu(),
        name='model',
        registered_model_name=f'DNN_TFIDF_{MEMBER}'
    )

    model_final.to(DEVICE)

    print('=' * 55)
    print('  Run registrado en MLflow')
    print(f'  Servidor    : {MLFLOW_URI}')
    print('  Experimento : Parcial_1_NLP')
    print(f'  Corrida     : DNN_TFIDF_{MEMBER}')
    print(f'  Run ID      : {run.info.run_id}')
    print(f'  Test F1     : {f1:.4f}')
    print(f'  Test AUC    : {auc:.4f}')
    print('=' * 55)